# 08 — FLORES-200 Out-of-Domain Validation

Phase 1 selects language-specific SAE features using monolinguality metrics and a language probe, both computed exclusively on MGSM math word problems. Phase 2b then ablates these features and classifies them (LANGUAGE / SHARED / REASONING / JUNK). The concern: features might be MGSM-specific artifacts rather than genuinely language-encoding.

This notebook validates that the selected features fire preferentially on their target language using completely out-of-domain parallel text from FLORES-200 (1012 professional translations of the *same* sentences across 200+ languages). Since content is held constant, any activation differences across languages must be due to language encoding, not topic or content.

**Five validation metrics:**
- V1: Monolinguality correlation (MGSM vs FLORES)
- V2: Target-language activation ratio + Cohen's d
- V3: Probe transfer accuracy (train on MGSM, test on FLORES)
- V4: LANGUAGE vs SHARED causal-tag separation on FLORES
- V5: Per-feature activation heatmap

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

DRIVE_RESULTS = None

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/kvrancic/nlp-project.git'

if IN_COLAB:
    REPO_DIR = '/content/nlp-project'
    if not Path(REPO_DIR).exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
    os.chdir(REPO_DIR)
    !pip install -q 'numpy>=2.0' datasets -e .
else:
    REPO_DIR = os.getcwd()  # local: assume cwd is the repo root

# Evict cached src.* so re-imports pick up the latest code
for _mod_name in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_mod_name]

if IN_COLAB:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    Path('.env').write_text(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')
else:
    from dotenv import load_dotenv
    load_dotenv()
    assert os.environ.get('HF_TOKEN'), 'HF_TOKEN missing in .env at repo root'

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_RESULTS = Path('/content/drive/MyDrive/nlp-project-results')
        DRIVE_RESULTS.mkdir(exist_ok=True, parents=True)
    except Exception:
        DRIVE_RESULTS = None

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm

from src.config import (
    TARGET_LANGUAGES, SAE_WIDTH_16K, RESULTS_DIR, D_MODEL,
)
from src.model import load_model_and_tokenizer, load_sae
from src.extraction import extract_residual_activations, encode_activations_through_sae
from src.monolinguality import compute_monolinguality, train_language_probe

torch.manual_seed(0)
np.random.seed(0)

PRIMARY_LAYER = 17  # layer where causal labeling happened

## 1. Load model, SAE, Phase 1 & Phase 2b results

In [ ]:
model, tokenizer = load_model_and_tokenizer()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sae, _, _ = load_sae(
    layer=PRIMARY_LAYER,
    width=SAE_WIDTH_16K,
    l0_target='medium',
)
print(f'SAE loaded: layer {PRIMARY_LAYER}, width {SAE_WIDTH_16K}')

# Phase 1
phase1 = torch.load(RESULTS_DIR / 'phase1_features.pt', weights_only=False)
intersection = phase1['intersection_features']  # {layer: {lang: [feat_indices]}}
mono_mgsm = phase1.get('monolinguality', None)  # {layer: {lang: tensor}}
print('Phase 1 loaded.')
for lang in TARGET_LANGUAGES:
    n = len(intersection[PRIMARY_LAYER][lang])
    print(f'  {lang}: {n} A\u2229B features at layer {PRIMARY_LAYER}')

# Phase 2b
phase2b = torch.load(RESULTS_DIR / 'phase2_ablation.pt', weights_only=False)
causal_labels = phase2b['causal_labels']  # {(lang, feat): {'tag': ..., ...}}
confirmed_language = phase2b['confirmed_language']  # {lang: [feat_indices]}
print('\nPhase 2b loaded.')
from collections import Counter
tag_counts = Counter(v['tag'] for v in causal_labels.values())
print(f'  Causal tags: {dict(tag_counts)}')

## 2. Load FLORES-200 data

In [ ]:
from datasets import load_dataset

# FLORES-200 language code mapping
FLORES_LANG_MAP = {
    'en': 'eng_Latn',
    'zh': 'zho_Hans',
    'es': 'spa_Latn',
    'bn': 'ben_Beng',
    'sw': 'swh_Latn',
}

flores_texts = {}
for lang, flores_code in FLORES_LANG_MAP.items():
    ds = load_dataset('facebook/flores', flores_code, split='devtest')
    flores_texts[lang] = ds['sentence']
    print(f'  {lang} ({flores_code}): {len(flores_texts[lang])} sentences')

print(f'\nSample (en): {flores_texts["en"][0][:100]}...')
print(f'Sample (zh): {flores_texts["zh"][0][:100]}...')

## 3. Extraction: FLORES activations through SAE

In [ ]:
# Wrap FLORES sentences in Gemma chat template (same format as MGSM)
# to keep activation distributions comparable
def make_chat_prompt(text):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': text}],
        tokenize=False,
        add_generation_prompt=True,
    )

flores_prompts = {
    lang: [make_chat_prompt(s) for s in sents]
    for lang, sents in flores_texts.items()
}
print(f'Total FLORES prompts: {sum(len(v) for v in flores_prompts.values())}')

In [ ]:
# Extract residual activations at layer 17 for all FLORES sentences
flores_residuals = {}  # lang -> (n_sentences, d_model)
for lang in TARGET_LANGUAGES:
    print(f'\nExtracting {lang} ({len(flores_prompts[lang])} sentences)...')
    acts = extract_residual_activations(
        model, tokenizer,
        flores_prompts[lang],
        layers=[PRIMARY_LAYER],
        positions='last',
    )
    flores_residuals[lang] = acts[PRIMARY_LAYER]
    print(f'  {lang}: {flores_residuals[lang].shape}')

# Encode through SAE
flores_feats = {}  # lang -> (n_sentences, n_sae_features)
for lang in TARGET_LANGUAGES:
    flores_feats[lang] = encode_activations_through_sae(
        flores_residuals[lang], sae,
    )
    print(f'  {lang} SAE features: {flores_feats[lang].shape}')

# Free residuals to save memory
del flores_residuals
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Re-extract MGSM features at layer 17 (needed for probe training in V3;
# Phase 1 artifact doesn't save raw feature_acts)
from src.data import load_mgsm

mgsm = load_mgsm(TARGET_LANGUAGES)
mgsm_prompts = {
    lang: [make_chat_prompt(ex['question']) for ex in mgsm[lang]]
    for lang in TARGET_LANGUAGES
}

mgsm_feats = {}  # lang -> (n_examples, n_sae_features)
for lang in TARGET_LANGUAGES:
    print(f'\nExtracting MGSM {lang} ({len(mgsm_prompts[lang])} prompts)...')
    acts = extract_residual_activations(
        model, tokenizer,
        mgsm_prompts[lang],
        layers=[PRIMARY_LAYER],
        positions='last',
    )
    mgsm_feats[lang] = encode_activations_through_sae(
        acts[PRIMARY_LAYER], sae,
    )
    print(f'  {lang} MGSM SAE features: {mgsm_feats[lang].shape}')
    del acts

torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 4. V1 — Monolinguality correlation (MGSM vs FLORES)

In [ ]:
# Compute monolinguality on FLORES
mono_flores = compute_monolinguality(flores_feats)

# Recompute monolinguality on MGSM (in case phase1 artifact format differs)
mono_mgsm_recomp = compute_monolinguality(mgsm_feats)

# Correlation per language
v1_results = {}
print('V1 — Monolinguality correlation (nu_MGSM vs nu_FLORES):')
print(f'{"lang":>4}  {"Pearson r":>10}  {"Spearman rho":>12}  {"pass?":>6}')
print('-' * 40)
for lang in TARGET_LANGUAGES:
    nu_mgsm = mono_mgsm_recomp[lang].numpy()
    nu_flores = mono_flores[lang].numpy()
    pearson_r, pearson_p = stats.pearsonr(nu_mgsm, nu_flores)
    spearman_rho, spearman_p = stats.spearmanr(nu_mgsm, nu_flores)
    passed = pearson_r > 0.5
    v1_results[lang] = {
        'pearson_r': float(pearson_r),
        'pearson_p': float(pearson_p),
        'spearman_rho': float(spearman_rho),
        'spearman_p': float(spearman_p),
        'pass': passed,
    }
    status = 'PASS' if passed else 'FAIL'
    print(f'{lang:>4}  {pearson_r:>10.4f}  {spearman_rho:>12.4f}  {status:>6}')

## 5. V2 — Target-language activation ratio

In [ ]:
v2_results = {}
print('V2 — Target-language activation ratio for A\u2229B features on FLORES:')
print(f'{"lang":>4}  {"n_feats":>7}  {"target_mean":>11}  {"other_mean":>10}  '
      f'{"ratio":>6}  {"Cohen d":>8}  {"pass?":>6}')
print('-' * 65)

for lang in TARGET_LANGUAGES:
    feat_indices = intersection[PRIMARY_LAYER][lang]
    if not feat_indices:
        v2_results[lang] = {'ratio': 0, 'cohens_d': 0, 'pass': False}
        print(f'{lang:>4}  no A\u2229B features')
        continue

    idx = torch.tensor(feat_indices)

    # Mean activation on target language
    target_acts = flores_feats[lang][:, idx].float()  # (n_sentences, n_feats)
    target_mean_per_sent = target_acts.mean(dim=1)  # (n_sentences,)

    # Mean activation across other 4 languages
    other_langs = [l for l in TARGET_LANGUAGES if l != lang]
    other_acts_list = [flores_feats[l][:, idx].float().mean(dim=1) for l in other_langs]
    other_mean_per_sent = torch.stack(other_acts_list).mean(dim=0)  # (n_sentences,)

    target_mean = target_mean_per_sent.mean().item()
    other_mean = other_mean_per_sent.mean().item()
    ratio = target_mean / max(other_mean, 1e-8)

    # Cohen's d
    pooled_std = torch.sqrt(
        (target_mean_per_sent.var() + other_mean_per_sent.var()) / 2
    ).item()
    cohens_d = (target_mean - other_mean) / max(pooled_std, 1e-8)

    passed = ratio > 2.0 and cohens_d > 0.5
    v2_results[lang] = {
        'n_feats': len(feat_indices),
        'target_mean': target_mean,
        'other_mean': other_mean,
        'ratio': ratio,
        'cohens_d': cohens_d,
        'pass': passed,
    }
    status = 'PASS' if passed else 'FAIL'
    print(f'{lang:>4}  {len(feat_indices):>7}  {target_mean:>11.4f}  {other_mean:>10.4f}  '
          f'{ratio:>6.2f}  {cohens_d:>8.2f}  {status:>6}')

## 6. V3 — Probe transfer accuracy (train MGSM, test FLORES)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

# Train probe on MGSM features
print('Training language probe on MGSM SAE features...')
probe, importances = train_language_probe(mgsm_feats)
languages_sorted = sorted(TARGET_LANGUAGES)

# In-sample accuracy on MGSM
X_mgsm = np.concatenate([mgsm_feats[l].float().numpy() for l in languages_sorted])
y_mgsm = np.concatenate([np.full(mgsm_feats[l].shape[0], i)
                          for i, l in enumerate(languages_sorted)])
mgsm_acc = accuracy_score(y_mgsm, probe.predict(X_mgsm))
print(f'  MGSM in-sample accuracy: {mgsm_acc:.4f}')

# Transfer: predict on FLORES features
X_flores = np.concatenate([flores_feats[l].float().numpy() for l in languages_sorted])
y_flores = np.concatenate([np.full(flores_feats[l].shape[0], i)
                            for i, l in enumerate(languages_sorted)])
y_pred_flores = probe.predict(X_flores)
transfer_acc = accuracy_score(y_flores, y_pred_flores)
cm = confusion_matrix(y_flores, y_pred_flores)

passed = transfer_acc > 0.80
status = 'PASS' if passed else 'FAIL'
print(f'  FLORES transfer accuracy: {transfer_acc:.4f}  [{status}]')
print(f'\nConfusion matrix (rows=true, cols=pred):')
print(f'  langs: {languages_sorted}')
print(cm)

v3_results = {
    'mgsm_accuracy': float(mgsm_acc),
    'transfer_accuracy': float(transfer_acc),
    'confusion_matrix': cm.tolist(),
    'languages': languages_sorted,
    'pass': passed,
}

## 7. V4 — LANGUAGE vs SHARED tag separation on FLORES

In [ ]:
# For each causally labeled feature, compute its FLORES monolinguality
# (for the language it was labeled under)
tag_nu_flores = {'LANGUAGE': [], 'SHARED': [], 'REASONING': [], 'JUNK': []}

for (lang, feat), info in causal_labels.items():
    tag = info['tag']
    nu = mono_flores[lang][feat].item()
    tag_nu_flores[tag].append(nu)

print('V4 — FLORES monolinguality by causal tag:')
for tag in ['LANGUAGE', 'SHARED', 'REASONING', 'JUNK']:
    vals = tag_nu_flores[tag]
    if vals:
        print(f'  {tag:>10}: n={len(vals):>3}, '
              f'mean={np.mean(vals):.4f}, median={np.median(vals):.4f}')
    else:
        print(f'  {tag:>10}: n=0')

# Mann-Whitney U test: LANGUAGE vs SHARED
lang_vals = tag_nu_flores['LANGUAGE']
shared_vals = tag_nu_flores['SHARED']

if len(lang_vals) >= 2 and len(shared_vals) >= 2:
    u_stat, u_pval = stats.mannwhitneyu(
        lang_vals, shared_vals, alternative='greater',
    )
    passed = u_pval < 0.01
    status = 'PASS' if passed else 'FAIL'
    print(f'\n  Mann-Whitney U (LANGUAGE > SHARED): U={u_stat:.1f}, p={u_pval:.4e}  [{status}]')
elif len(lang_vals) < 2:
    u_stat, u_pval = float('nan'), float('nan')
    passed = False
    print('\n  Not enough LANGUAGE-tagged features for Mann-Whitney U test')
else:
    u_stat, u_pval = float('nan'), float('nan')
    passed = False
    print('\n  Not enough SHARED-tagged features for Mann-Whitney U test')

v4_results = {
    'tag_nu_flores': {k: v for k, v in tag_nu_flores.items()},
    'u_stat': float(u_stat) if not np.isnan(u_stat) else None,
    'u_pval': float(u_pval) if not np.isnan(u_pval) else None,
    'pass': passed,
}

## 8. V5 — Per-feature activation heatmap

In [ ]:
# Build matrix: rows = selected features (grouped by target language),
# columns = 5 evaluation languages, values = mean FLORES activation.
all_feats = []  # (target_lang, feat_index)
row_labels = []
for lang in TARGET_LANGUAGES:
    for feat in intersection[PRIMARY_LAYER][lang]:
        all_feats.append((lang, feat))
        row_labels.append(f'{lang}:{feat}')

heatmap_matrix = np.zeros((len(all_feats), len(TARGET_LANGUAGES)))
for i, (target_lang, feat_idx) in enumerate(all_feats):
    for j, eval_lang in enumerate(TARGET_LANGUAGES):
        heatmap_matrix[i, j] = flores_feats[eval_lang][:, feat_idx].float().mean().item()

print(f'Heatmap matrix shape: {heatmap_matrix.shape} '
      f'(features x eval_languages)')

v5_results = {
    'heatmap_matrix': heatmap_matrix.tolist(),
    'row_labels': row_labels,
    'col_labels': TARGET_LANGUAGES,
    'feature_info': all_feats,
}

## 9. Figures

In [ ]:
FIG_DIR = Path('results/figures')
FIG_DIR.mkdir(exist_ok=True, parents=True)
sns.set_theme(style='whitegrid', context='paper')

# Figure 1: Scatter plots — nu_MGSM vs nu_FLORES per language (top-200 by |nu|)
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
for ax, lang in zip(axes, TARGET_LANGUAGES):
    nu_m = mono_mgsm_recomp[lang].numpy()
    nu_f = mono_flores[lang].numpy()
    # Select top-200 features by |nu_MGSM|
    top_idx = np.argsort(np.abs(nu_m))[-200:]
    ax.scatter(nu_m[top_idx], nu_f[top_idx], s=8, alpha=0.5)
    # Regression line
    z = np.polyfit(nu_m[top_idx], nu_f[top_idx], 1)
    x_line = np.linspace(nu_m[top_idx].min(), nu_m[top_idx].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'r-', linewidth=1.5)
    r = v1_results[lang]['pearson_r']
    ax.set_title(f'{lang} (r={r:.3f})')
    ax.set_xlabel('$\\nu_{MGSM}$')
    if ax == axes[0]:
        ax.set_ylabel('$\\nu_{FLORES}$')
fig.suptitle('V1: Monolinguality correlation (top-200 features by |$\\nu$|)', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_v1_mono_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Bar chart — target vs other activation for each language's features
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(TARGET_LANGUAGES))
width = 0.35
target_means = [v2_results[l].get('target_mean', 0) for l in TARGET_LANGUAGES]
other_means = [v2_results[l].get('other_mean', 0) for l in TARGET_LANGUAGES]
ax.bar(x - width/2, target_means, width, label='Target language', color='C0')
ax.bar(x + width/2, other_means, width, label='Other languages (avg)', color='C1')
ax.set_xticks(x)
ax.set_xticklabels(TARGET_LANGUAGES)
ax.set_ylabel('Mean SAE feature activation')
ax.set_title('V2: Target vs other language activation (A$\\cap$B features on FLORES)')
ax.legend()
# Annotate ratios
for i, lang in enumerate(TARGET_LANGUAGES):
    ratio = v2_results[lang].get('ratio', 0)
    ax.annotate(f'{ratio:.1f}x', (i, max(target_means[i], other_means[i])),
                ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_v2_target_ratio.png', dpi=150)
plt.show()

In [ ]:
# Figure 3: Confusion matrix heatmap from probe transfer
fig, ax = plt.subplots(figsize=(6, 5))
cm_array = np.array(v3_results['confusion_matrix'])
# Normalize by row (true label)
cm_norm = cm_array.astype(float) / cm_array.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=languages_sorted, yticklabels=languages_sorted, ax=ax)
ax.set_xlabel('Predicted language')
ax.set_ylabel('True language')
ax.set_title(f'V3: Probe transfer (MGSM$\\rightarrow$FLORES)\n'
             f'Transfer accuracy: {v3_results["transfer_accuracy"]:.3f}')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_v3_probe_transfer_cm.png', dpi=150)
plt.show()

In [ ]:
# Figure 4: Box/violin — FLORES nu grouped by causal tag
fig, ax = plt.subplots(figsize=(7, 4))
plot_data = []
for tag in ['LANGUAGE', 'SHARED', 'REASONING', 'JUNK']:
    for val in tag_nu_flores[tag]:
        plot_data.append({'tag': tag, 'nu_flores': val})
if plot_data:
    df_plot = pd.DataFrame(plot_data)
    tag_order = [t for t in ['LANGUAGE', 'SHARED', 'REASONING', 'JUNK']
                 if t in df_plot['tag'].values]
    sns.violinplot(data=df_plot, x='tag', y='nu_flores', order=tag_order,
                   inner='box', ax=ax, cut=0)
    sns.stripplot(data=df_plot, x='tag', y='nu_flores', order=tag_order,
                  color='black', size=4, alpha=0.6, ax=ax)
    ax.set_ylabel('$\\nu_{FLORES}$')
    ax.set_xlabel('Causal tag (Phase 2b)')
    pval_str = f'p={v4_results["u_pval"]:.2e}' if v4_results['u_pval'] is not None else 'N/A'
    ax.set_title(f'V4: FLORES monolinguality by causal tag\n'
                 f'Mann-Whitney U (LANGUAGE > SHARED): {pval_str}')
else:
    ax.text(0.5, 0.5, 'No causal labels available', ha='center', va='center',
            transform=ax.transAxes)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_v4_tag_separation.png', dpi=150)
plt.show()

In [ ]:
# Figure 5: Feature x language activation heatmap
if len(all_feats) > 0:
    fig, ax = plt.subplots(figsize=(8, max(6, len(all_feats) * 0.25)))
    sns.heatmap(
        heatmap_matrix, ax=ax, cmap='YlOrRd',
        xticklabels=TARGET_LANGUAGES,
        yticklabels=row_labels,
        cbar_kws={'label': 'Mean activation'},
    )
    # Add horizontal lines between language groups
    cumulative = 0
    for lang in TARGET_LANGUAGES:
        n_lang_feats = sum(1 for tl, _ in all_feats if tl == lang)
        cumulative += n_lang_feats
        if cumulative < len(all_feats):
            ax.axhline(y=cumulative, color='white', linewidth=2)
    ax.set_title('V5: Feature activation heatmap (FLORES)\n'
                 'Rows = A$\\cap$B features grouped by target language')
    ax.set_ylabel('Feature (target_lang:feat_idx)')
    ax.set_xlabel('Evaluation language')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig_v5_activation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No features to plot.')

## 10. Save results

In [ ]:
# Summary table
summary_rows = []
for lang in TARGET_LANGUAGES:
    summary_rows.append({
        'language': lang,
        'V1_pearson_r': v1_results[lang]['pearson_r'],
        'V1_pass': v1_results[lang]['pass'],
        'V2_ratio': v2_results[lang].get('ratio', None),
        'V2_cohens_d': v2_results[lang].get('cohens_d', None),
        'V2_pass': v2_results[lang].get('pass', False),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df['V3_transfer_acc'] = v3_results['transfer_accuracy']
summary_df['V3_pass'] = v3_results['pass']
summary_df['V4_pass'] = v4_results['pass']
print('\n=== Validation Summary ===')
print(summary_df.to_string(index=False))

# Overall verdict
all_v1_pass = all(v1_results[l]['pass'] for l in TARGET_LANGUAGES)
all_v2_pass = all(v2_results[l].get('pass', False) for l in TARGET_LANGUAGES)
print(f'\nV1 (all langs r>0.5): {"PASS" if all_v1_pass else "FAIL"}')
print(f'V2 (all langs ratio>2, d>0.5): {"PASS" if all_v2_pass else "FAIL"}')
print(f'V3 (transfer acc>0.80): {"PASS" if v3_results["pass"] else "FAIL"} '
      f'({v3_results["transfer_accuracy"]:.3f})')
print(f'V4 (LANGUAGE > SHARED, p<0.01): {"PASS" if v4_results["pass"] else "FAIL"}')
print(f'V5: see heatmap above')

# Save
payload = {
    'config': {
        'primary_layer': PRIMARY_LAYER,
        'sae_width': SAE_WIDTH_16K,
        'n_flores_sentences': len(flores_texts['en']),
        'flores_lang_map': FLORES_LANG_MAP,
    },
    'v1_monolinguality_correlation': v1_results,
    'v2_target_activation_ratio': v2_results,
    'v3_probe_transfer': v3_results,
    'v4_tag_separation': v4_results,
    'v5_activation_heatmap': v5_results,
    'mono_flores': {lang: mono_flores[lang].numpy().tolist() for lang in TARGET_LANGUAGES},
    'summary': summary_df.to_dict(orient='records'),
}

out = RESULTS_DIR / 'phase_validation_flores.pt'
torch.save(payload, out)
print(f'\nSaved {out}')

if DRIVE_RESULTS is not None:
    torch.save(payload, DRIVE_RESULTS / 'phase_validation_flores.pt')
    print(f'Backed up to {DRIVE_RESULTS / "phase_validation_flores.pt"}')